In [4]:
import pandas as pd

# Load datasets
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# Check datasets
print("Train shape:", train.shape)
print("Test shape:", test.shape)

display(train.head())
display(test.head())

print("Fraud rate:", train["is_fraud"].mean())

Train shape: (20000, 11)
Test shape: (10000, 10)


,claim_id,claim_amount,driver_age,previous_claim_count,vehicle_value,damage_severity,incident_type,repair_shop_id,phone_hash,address_id,is_fraud
0,C000001,254.29,42.0,2,11900.0,low,collision,Sb7a282e399,Pfd09f60f9e,A48656b5517,0
1,C000002,592.05,29.0,0,14600.0,low,collision,S9399315447,P8a7ff077bb,A53e94c25e5,0
2,C000007,980.14,58.0,0,17550.0,low,glass,S1bb582e486,P02246ac9cc,A84ed465d68,0
3,C000008,729.04,44.0,0,18550.0,low,collision,S7e56cdce9e,P8e162b8d1a,A46a2d8b11d,0
4,C000009,16145.05,34.0,0,29300.0,NaN,theft,S1bb582e486,Pd299199a4f,Ac638621b9b,0


,claim_id,claim_amount,driver_age,previous_claim_count,vehicle_value,damage_severity,incident_type,repair_shop_id,phone_hash,address_id
0,C000003,757.53,29.0,0,19700.0,medium,glass,S9dd55af075,Pf78d7edda7,Ae9e2783b3d
1,C000004,903.64,47.0,0,16450.0,low,collision,S71285760c2,P76b9d13e6a,A2dd867ebb2
2,C000005,20604.38,59.0,0,24750.0,high,theft,S1bb582e486,P7d5a46ed58,A659da9fba9
3,C000006,710.33,30.0,0,19650.0,low,collision,S1bb582e486,P41a8d51c6a,Ad4d7ace23a
4,C000010,6674.53,37.0,0,20600.0,high,collision,S19815489c9,P430e77acfb,Aafa83427b8


Fraud rate: 0.1717


In [9]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier
import numpy as np

# Prepare features
X = train.drop(columns=["is_fraud", "claim_id"]).copy()
y = train["is_fraud"]

cat_cols = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

for col in cat_cols:
    X[col] = X[col].fillna("Missing").astype(str)

# 5-fold cross-validation
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_probs = np.zeros(len(train))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):

    model_cv = CatBoostClassifier(
        iterations=303,
        depth=6,
        learning_rate=0.05,
        loss_function="Logloss",
        verbose=False,
        random_seed=42,
        cat_features=cat_cols
    )

    model_cv.fit(
        X.iloc[train_idx],
        y.iloc[train_idx]
    )

    oof_probs[val_idx] = model_cv.predict_proba(
        X.iloc[val_idx]
    )[:, 1]

    print(f"Fold {fold} completed")

# Calculate expected savings
amounts = train["claim_amount"].to_numpy()
fraud = y.to_numpy()

expected_savings = oof_probs * amounts - 2500

# Scale investigation budget to 20,000 claims
budget = 4000

investigate = np.zeros(len(train), dtype=int)

top_indices = np.argsort(-expected_savings)[:budget]
top_indices = top_indices[expected_savings[top_indices] > 0]

investigate[top_indices] = 1

# Calculate S score
S = np.sum(
    investigate * (fraud * amounts - 2500)
)

print("\n===== CROSS-VALIDATION RESULTS =====")
print(f"ROC-AUC: {roc_auc_score(y, oof_probs):.4f}")
print(f"Claims investigated: {investigate.sum()}")
print(f"Frauds detected: {np.sum(investigate * fraud)}")
print(f"S Score: ${S:,.2f}")
print(f"Estimated S for 10,000 claims: ${S / 2:,.2f}")

Fold 1 completed
Fold 2 completed
Fold 3 completed
Fold 4 completed
Fold 5 completed

===== CROSS-VALIDATION RESULTS =====
ROC-AUC: 0.8036
Claims investigated: 2406
Frauds detected: 982
S Score: $11,944,392.08
Estimated S for 10,000 claims: $5,972,196.04


In [11]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# ==========================================
# 1. Prepare data
# ==========================================

X = train.drop(columns=["is_fraud", "claim_id"]).copy()
y = train["is_fraud"]

cat_cols = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

for col in cat_cols:
    X[col] = X[col].fillna("Missing").astype(str)


# ==========================================
# 2. Feature engineering
# ==========================================

def add_features(X_train_fold, X_val_fold):

    X_train_fold = X_train_fold.copy()
    X_val_fold = X_val_fold.copy()

    # Feature 1: Claim-to-vehicle value ratio
    for df in [X_train_fold, X_val_fold]:
        df["claim_vehicle_ratio"] = (
            df["claim_amount"] /
            df["vehicle_value"].replace(0, np.nan)
        )

    # Features 2-4: Identifier frequencies
    for col in ["phone_hash", "repair_shop_id", "address_id"]:

        # Learn frequencies from training fold only
        counts = X_train_fold[col].value_counts()

        X_train_fold[col + "_frequency"] = (
            X_train_fold[col].map(counts).fillna(0)
        )

        X_val_fold[col + "_frequency"] = (
            X_val_fold[col].map(counts).fillna(0)
        )

    return X_train_fold, X_val_fold


# ==========================================
# 3. 5-fold cross-validation
# ==========================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_probs_fe = np.zeros(len(train))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):

    X_tr = X.iloc[train_idx]
    X_va = X.iloc[val_idx]

    y_tr = y.iloc[train_idx]

    # Add engineered features
    X_tr, X_va = add_features(X_tr, X_va)

    # Train CatBoost
    model_fe = CatBoostClassifier(
        iterations=303,
        depth=6,
        learning_rate=0.05,
        loss_function="Logloss",
        verbose=False,
        random_seed=42,
        cat_features=cat_cols
    )

    model_fe.fit(X_tr, y_tr)

    # Predict validation probabilities
    oof_probs_fe[val_idx] = model_fe.predict_proba(
        X_va
    )[:, 1]

    print(f"Fold {fold} completed")


# ==========================================
# 4. Calculate competition S score
# ==========================================

amounts = train["claim_amount"].to_numpy()
fraud = y.to_numpy()

# Expected savings
expected_savings = oof_probs_fe * amounts - 2500

# Maximum 20% of claims investigated
budget = 4000

investigate = np.zeros(len(train), dtype=int)

top_indices = np.argsort(-expected_savings)[:budget]

# Only investigate positive expected savings
top_indices = top_indices[
    expected_savings[top_indices] > 0
]

investigate[top_indices] = 1

# Calculate actual financial score
S = np.sum(
    investigate * (fraud * amounts - 2500)
)

# ==========================================
# 5. Display results
# ==========================================

print("\n===== FEATURE ENGINEERING RESULTS =====")

print(f"ROC-AUC: {roc_auc_score(y, oof_probs_fe):.4f}")
print(f"Claims investigated: {investigate.sum()}")
print(f"Frauds detected: {np.sum(investigate * fraud)}")
print(f"S Score: ${S:,.2f}")
print(f"Estimated S for 10,000 claims: ${S / 2:,.2f}")

Fold 1 completed
Fold 2 completed
Fold 3 completed
Fold 4 completed
Fold 5 completed

===== FEATURE ENGINEERING RESULTS =====
ROC-AUC: 0.8117
Claims investigated: 2201
Frauds detected: 968
S Score: $12,190,989.48
Estimated S for 10,000 claims: $6,095,494.74


In [12]:
# ==========================================
# 1. Prepare full training and test data
# ==========================================

X_full = train.drop(columns=["is_fraud", "claim_id"]).copy()
X_test = test.drop(columns=["claim_id"]).copy()

y_full = train["is_fraud"]

# Handle categorical missing values
for col in cat_cols:
    X_full[col] = X_full[col].fillna("Missing").astype(str)
    X_test[col] = X_test[col].fillna("Missing").astype(str)

# ==========================================
# 2. Apply feature engineering
# ==========================================

X_full, X_test = add_features(X_full, X_test)

# ==========================================
# 3. Train final CatBoost model
# ==========================================

final_model = CatBoostClassifier(
    iterations=303,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    verbose=100,
    random_seed=42,
    cat_features=cat_cols
)

final_model.fit(X_full, y_full)

# ==========================================
# 4. Predict test fraud probabilities
# ==========================================

test_probs = final_model.predict_proba(X_test)[:, 1]

# ==========================================
# 5. Calculate expected financial savings
# ==========================================

expected_savings = (
    test_probs * test["claim_amount"].to_numpy() - 2500
)

# Select at most 2000 claims
budget = 2000

investigate = np.zeros(len(test), dtype=int)

top_indices = np.argsort(-expected_savings)[:budget]

top_indices = top_indices[
    expected_savings[top_indices] > 0
]

investigate[top_indices] = 1

# ==========================================
# 6. Generate submission
# ==========================================

submission = pd.DataFrame({
    "claim_id": test["claim_id"],
    "investigate": investigate
})

submission.to_csv("submission.csv", index=False)

print("\n===== FINAL SUBMISSION =====")
print("Submission shape:", submission.shape)
print("Claims investigated:", submission["investigate"].sum())
print("Unique claim IDs:", submission["claim_id"].nunique())

display(submission.head())

0:	learn: 0.6578068	total: 16.5ms	remaining: 4.97s
100:	learn: 0.3448270	total: 2.49s	remaining: 4.97s
200:	learn: 0.3320455	total: 4.57s	remaining: 2.32s
300:	learn: 0.3221649	total: 6.72s	remaining: 44.7ms
302:	learn: 0.3220314	total: 6.76s	remaining: 0us

===== FINAL SUBMISSION =====
Submission shape: (10000, 2)
Claims investigated: 998
Unique claim IDs: 10000


,claim_id,investigate
0,C000003,0
1,C000004,0
2,C000005,0
3,C000006,0
4,C000010,0


In [14]:
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

configs = [
    {"iterations": 500, "depth": 5, "learning_rate": 0.05},
    {"iterations": 500, "depth": 7, "learning_rate": 0.05},
    {"iterations": 700, "depth": 6, "learning_rate": 0.03}
]

results = []
cv_predictions = {}

amounts = train["claim_amount"].to_numpy()
fraud = train["is_fraud"].to_numpy()

for config in configs:

    print(f"\nTesting: {config}")

    oof = np.zeros(len(train))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):

        X_tr, X_va = add_features(
            X.iloc[tr_idx],
            X.iloc[va_idx]
        )

        model_tuned = CatBoostClassifier(
            **config,
            loss_function="Logloss",
            verbose=False,
            random_seed=42,
            cat_features=cat_cols
        )

        model_tuned.fit(X_tr, y.iloc[tr_idx])

        oof[va_idx] = model_tuned.predict_proba(X_va)[:, 1]

    # Use original threshold for fair comparison
    expected_value = oof * amounts

    selected = np.where(expected_value > 2500)[0]
    selected = selected[np.argsort(-expected_value[selected])][:4000]

    S = np.sum(fraud[selected] * amounts[selected] - 2500)

    name = (
        f"iter={config['iterations']}, "
        f"depth={config['depth']}, "
        f"lr={config['learning_rate']}"
    )

    cv_predictions[name] = oof.copy()

    results.append({
        "Model": name,
        "ROC-AUC": roc_auc_score(y, oof),
        "Investigated": len(selected),
        "S Score": S,
        "Estimated Test S": S / 2
    })

    print(f"S Score: ${S:,.2f}")

results_df = pd.DataFrame(results).sort_values(
    "S Score", ascending=False
)

display(results_df)


Testing: {'iterations': 500, 'depth': 5, 'learning_rate': 0.05}
S Score: $12,135,340.06

Testing: {'iterations': 500, 'depth': 7, 'learning_rate': 0.05}
S Score: $12,154,101.87

Testing: {'iterations': 700, 'depth': 6, 'learning_rate': 0.03}
S Score: $12,215,667.18


,Model,ROC-AUC,Investigated,S Score,Estimated Test S
2,"iter=700, depth=6, lr=0.03",0.812177,2177,12215667.18,6107833.590
1,"iter=500, depth=7, lr=0.05",0.810116,2131,12154101.87,6077050.935
0,"iter=500, depth=5, lr=0.05",0.811710,2191,12135340.06,6067670.030


In [15]:
import numpy as np
import pandas as pd

amounts = train["claim_amount"].to_numpy()
fraud = train["is_fraud"].to_numpy()

thresholds = np.arange(2500, 50001, 2500)

results = []

for threshold in thresholds:

    # Select claims above the threshold
    candidates = np.where(amounts >= threshold)[0]

    # Prioritise the largest claims
    selected = candidates[
        np.argsort(-amounts[candidates])
    ][:4000]

    # Calculate financial score
    S = np.sum(
        fraud[selected] * amounts[selected] - 2500
    )

    results.append({
        "Claim Threshold": threshold,
        "Investigated": len(selected),
        "Frauds Detected": fraud[selected].sum(),
        "S Score": S,
        "Estimated Test S": S / 2
    })

amount_results = pd.DataFrame(results)

display(
    amount_results.sort_values(
        "S Score", ascending=False
    ).head(10)
)

,Claim Threshold,Investigated,Frauds Detected,S Score,Estimated Test S
3,10000,3449,872,9864472.17,4932236.085
4,12500,2733,712,9849064.72,4924532.360
0,2500,4000,987,9539198.88,4769599.440
1,5000,4000,987,9539198.88,4769599.440
2,7500,4000,987,9539198.88,4769599.440
5,15000,2097,564,9399304.52,4699652.260
6,17500,1551,423,8480164.82,4240082.410
7,20000,1176,334,7754646.44,3877323.220
8,22500,897,265,6995995.05,3497997.525
9,25000,665,201,6065399.75,3032699.875


In [17]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# ==========================================
# 1. Retrieve existing predictions
# ==========================================

p_original = oof_probs_fe

p_500_d5 = cv_predictions[
    "iter=500, depth=5, lr=0.05"
]

p_500_d7 = cv_predictions[
    "iter=500, depth=7, lr=0.05"
]

p_700_d6 = cv_predictions[
    "iter=700, depth=6, lr=0.03"
]

# ==========================================
# 2. Create ensembles
# ==========================================

ensembles = {
    "Original CatBoost": p_original,

    "700-iteration CatBoost": p_700_d6,

    "Equal Weight (4 models)": (
        p_original + p_500_d5 + p_500_d7 + p_700_d6
    ) / 4,

    "Best Two (50/50)": (
        p_original + p_700_d6
    ) / 2,

    "Weighted (30/70)": (
        0.3 * p_original + 0.7 * p_700_d6
    )
}

# ==========================================
# 3. Evaluate S score
# ==========================================

amounts = train["claim_amount"].to_numpy()
fraud = train["is_fraud"].to_numpy()

results = []

for name, probs in ensembles.items():

    # Expected fraud savings
    expected_value = probs * amounts

    # Investigate if expected savings exceed cost
    candidates = np.where(expected_value > 2500)[0]

    # Rank by expected savings
    selected = candidates[
        np.argsort(-expected_value[candidates])
    ][:4000]

    # Actual financial score
    S = np.sum(
        fraud[selected] * amounts[selected] - 2500
    )

    results.append({
        "Model": name,
        "ROC-AUC": roc_auc_score(fraud, probs),
        "Investigated": len(selected),
        "Frauds Detected": fraud[selected].sum(),
        "S Score": S,
        "Estimated Test S": S / 2
    })

results_df = pd.DataFrame(results).sort_values(
    "S Score", ascending=False
)

display(results_df)

,Model,ROC-AUC,Investigated,Frauds Detected,S Score,Estimated Test S
2,Equal Weight (4 models),0.812480,2182,973,12273221.97,6136610.985
3,Best Two (50/50),0.812380,2190,971,12233696.40,6116848.200
4,Weighted (30/70),0.812409,2190,973,12217196.89,6108598.445
1,700-iteration CatBoost,0.812177,2177,973,12215667.18,6107833.590
0,Original CatBoost,0.811692,2201,968,12190989.48,6095494.740


In [18]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_auc_score

oof_lgbm = np.zeros(len(train))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):

    # Feature engineering
    X_tr, X_va = add_features(
        X.iloc[tr_idx],
        X.iloc[va_idx]
    )

    # Encode categorical features
    encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )

    X_tr[cat_cols] = encoder.fit_transform(X_tr[cat_cols])
    X_va[cat_cols] = encoder.transform(X_va[cat_cols])

    # Train LightGBM
    model_lgbm = LGBMClassifier(
        n_estimators=500,
        learning_rate=0.03,
        num_leaves=15,
        max_depth=6,
        min_child_samples=30,
        random_state=42,
        verbosity=-1
    )

    model_lgbm.fit(X_tr, y.iloc[tr_idx])

    oof_lgbm[va_idx] = model_lgbm.predict_proba(X_va)[:, 1]

    print(f"Fold {fold} completed")

# Evaluate
amounts = train["claim_amount"].to_numpy()
fraud = train["is_fraud"].to_numpy()

expected_value = oof_lgbm * amounts

selected = np.where(expected_value > 2500)[0]
selected = selected[np.argsort(-expected_value[selected])][:4000]

S_lgbm = np.sum(fraud[selected] * amounts[selected] - 2500)

print("\n===== LIGHTGBM RESULTS =====")
print(f"ROC-AUC: {roc_auc_score(y, oof_lgbm):.4f}")
print(f"Investigated: {len(selected)}")
print(f"S Score: ${S_lgbm:,.2f}")
print(f"Estimated Test S: ${S_lgbm / 2:,.2f}")

Fold 1 completed
Fold 2 completed
Fold 3 completed
Fold 4 completed
Fold 5 completed

===== LIGHTGBM RESULTS =====
ROC-AUC: 0.7941
Investigated: 2005
S Score: $11,899,914.98
Estimated Test S: $5,949,957.49


In [19]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# Existing CatBoost ensemble
catboost_ensemble = (
    p_original + p_500_d5 + p_500_d7 + p_700_d6
) / 4

# Test different ensemble weights
weights = [0, 0.1, 0.2, 0.3, 0.4, 0.5]

results = []

amounts = train["claim_amount"].to_numpy()
fraud = train["is_fraud"].to_numpy()

for w in weights:

    # w = weight assigned to LightGBM
    probs = (
        (1 - w) * catboost_ensemble
        + w * oof_lgbm
    )

    expected_value = probs * amounts

    candidates = np.where(expected_value > 2500)[0]

    selected = candidates[
        np.argsort(-expected_value[candidates])
    ][:4000]

    S = np.sum(
        fraud[selected] * amounts[selected] - 2500
    )

    results.append({
        "LightGBM Weight": w,
        "CatBoost Weight": 1 - w,
        "ROC-AUC": roc_auc_score(fraud, probs),
        "Investigated": len(selected),
        "S Score": S,
        "Estimated Test S": S / 2
    })

blend_results = pd.DataFrame(results).sort_values(
    "S Score", ascending=False
)

display(blend_results)

,LightGBM Weight,CatBoost Weight,ROC-AUC,Investigated,S Score,Estimated Test S
2,0.2,0.8,0.812261,2141,12283010.31,6141505.155
4,0.4,0.6,0.810567,2120,12282442.99,6141221.495
3,0.3,0.7,0.811590,2140,12273292.39,6136646.195
0,0.0,1.0,0.812480,2182,12273221.97,6136610.985
1,0.1,0.9,0.812556,2163,12227859.93,6113929.965
5,0.5,0.5,0.809134,2101,12224522.55,6112261.275


In [20]:
def add_features(X_train_fold, X_val_fold):

    X_train_fold = X_train_fold.copy()
    X_val_fold = X_val_fold.copy()

    # 1. Claim-to-vehicle value ratio
    for df in [X_train_fold, X_val_fold]:
        df["claim_vehicle_ratio"] = (
            df["claim_amount"] /
            df["vehicle_value"].replace(0, np.nan)
        )

    # 2. Identifier frequencies
    for col in ["phone_hash", "repair_shop_id", "address_id"]:

        counts = X_train_fold[col].value_counts()

        X_train_fold[col + "_frequency"] = (
            X_train_fold[col].map(counts).fillna(0)
        )

        X_val_fold[col + "_frequency"] = (
            X_val_fold[col].map(counts).fillna(0)
        )

    # 3. High-frequency flags
    for df in [X_train_fold, X_val_fold]:

        df["high_phone_frequency"] = (
            df["phone_hash_frequency"] >= 3
        ).astype(int)

        df["high_address_frequency"] = (
            df["address_id_frequency"] >= 3
        ).astype(int)

    # 4. Phone-address combination frequency
    pair_cols = ["phone_hash", "address_id"]

    pair_counts = (
        X_train_fold
        .groupby(pair_cols)
        .size()
        .rename("phone_address_frequency")
        .reset_index()
    )

    X_train_fold = X_train_fold.merge(
        pair_counts,
        on=pair_cols,
        how="left",
        sort=False
    )

    X_val_fold = X_val_fold.merge(
        pair_counts,
        on=pair_cols,
        how="left",
        sort=False
    )

    X_val_fold["phone_address_frequency"] = (
        X_val_fold["phone_address_frequency"].fillna(0)
    )

    return X_train_fold, X_val_fold